In [13]:
import pickle
import pandas as pd
import numpy as np

# --------------------------------------------------------
# 1. LOAD MODEL + ENCODERS
# --------------------------------------------------------
with open("genetic_prediction_bundle.pkl", "rb") as f:
    bundle = pickle.load(f)

model = bundle["gb_model"]
le_rsid = bundle["le_rsid"]
le_genotype = bundle["le_genotype"]
le_prs_group = bundle["le_prs_group"]

# --------------------------------------------------------
# 2. LOAD AND FIX GWAS DATABASE
# --------------------------------------------------------
gwas = pd.read_csv("GWAS_clean_disease.csv")
gwas.columns = [c.lower() for c in gwas.columns]

disease_col = "disease"  # your column is DISEASE

# --------------------------------------------------------
# Convert SNP → Standard rsID Format
# --------------------------------------------------------
def convert_snp_to_rsid(val):
    """
    Converts GWAS SNP values into standardized rsID:
    - 11591147.0 → rs11591147
    - rs11591147 → rs11591147
    - other non-rs formats → None
    """
    if pd.isna(val):
        return None

    val_str = str(val).strip()

    # Case 1: Numeric float (11591147.0)
    if val_str.replace(".", "").isdigit():
        val_str = val_str.split(".")[0]
        return "rs" + val_str

    # Case 2: Already rsXXXXX
    if val_str.lower().startswith("rs"):
        return val_str.lower()

    # Case 3: Format like "1:55520994"
    return None

gwas["rsid_clean"] = gwas["snp"].apply(convert_snp_to_rsid)

# --------------------------------------------------------
# 3. NEW SNP INPUT SAMPLES
# --------------------------------------------------------
raw = pd.DataFrame({
    "rsid": ["rs11591147", "rs2225986", "rs36024104"],
    "chromosome": [1, 1, 1],
    "pos": [55516852, 2225986, 36024104],
    "genotype": ["GG", "AG", "AA"],     # low, medium, high genotype
    "BETA": [0.05, 0.35, 1.10]          
})

# normalize input rsid
raw["rsid"] = raw["rsid"].astype(str).str.lower().str.strip()

# --------------------------------------------------------
# 4. SAFE ENCODING FOR UNSEEN VALUES
# --------------------------------------------------------
def safe_encode(le, val):
    try:
        return le.transform([val])[0]
    except:
        return 0  # default if unseen

raw["rsid_encoded"] = raw["rsid"].apply(lambda x: safe_encode(le_rsid, x))
raw["genotype_encoded"] = raw["genotype"].apply(lambda x: safe_encode(le_genotype, x))

# --------------------------------------------------------
# 5. FEATURE MATRIX FOR MODEL
# --------------------------------------------------------
X_new = raw[["rsid_encoded", "chromosome", "pos", "genotype_encoded", "BETA"]]

# --------------------------------------------------------
# 6. MODEL PREDICTION
# --------------------------------------------------------
pred_encoded = model.predict(X_new)
pred_labels = le_prs_group.inverse_transform(pred_encoded)

raw["PRS_Group"] = pred_labels

# --------------------------------------------------------
# 7. DISEASE LOOKUP USING CLEANED rsID
# --------------------------------------------------------
def get_disease(rsid):
    row = gwas[gwas["rsid_clean"] == rsid]
    if len(row) > 0:
        return row.iloc[0][disease_col]
    else:
        return "Unknown / Not in GWAS"

raw["Associated_Disease"] = raw["rsid"].apply(get_disease)

# --------------------------------------------------------
# 8. PRINT FINAL RESULT
# --------------------------------------------------------
print("\nFINAL PREDICTION RESULTS:\n")
for i, row in raw.iterrows():
    print(f"RSID:                {row['rsid']}")
    print(f"  Genotype:          {row['genotype']}")
    print(f"  BETA Score:        {row['BETA']}")
    print(f"  PRS Risk Group:    {row['PRS_Group']}")
    print(f"  Disease:           {row['Associated_Disease']}")
    print("----------------------------------------------------")



FINAL PREDICTION RESULTS:

RSID:                rs11591147
  Genotype:          GG
  BETA Score:        0.05
  PRS Risk Group:    Low
  Disease:           Plasma proprotein convertase subtilisin/kexin type 9 levels in stable coronary artery disease
----------------------------------------------------
RSID:                rs2225986
  Genotype:          AG
  BETA Score:        0.35
  PRS Risk Group:    Medium
  Disease:           Spherical equivalent or myopia (age of diagnosis)
----------------------------------------------------
RSID:                rs36024104
  Genotype:          AA
  BETA Score:        1.1
  PRS Risk Group:    High
  Disease:           Spherical equivalent or myopia (age of diagnosis)
----------------------------------------------------
